In [ ]:
!pip install -q tensorflow tensorflow-model-optimization ultralytics

from google.colab import drive
drive.mount('/content/drive')

import os, tempfile, zipfile
import numpy as np
import tensorflow as tf
from tensorflow_model_optimization.python.core.keras.compat import keras
import tensorflow_model_optimization as tfmot

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow: 2.20.0
GPU: []


In [ ]:
DATASET_PATH = '/content/drive/MyDrive/FYP/fyp-preprocessed'
MODEL_SAVE_PATH = '/content/drive/MyDrive/FYP/fyp-model'
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

TRAIN_DIR = f'{DATASET_PATH}/train/images'
VAL_DIR   = f'{DATASET_PATH}/val/images'
TEST_DIR  = f'{DATASET_PATH}/test/images'

IMG_SIZE  = 320   # Same as preprocessing
BATCH     = 16
EPOCHS    = 30
NC        = 21    # Number of navigation classes

print(f"Train images: {len(os.listdir(TRAIN_DIR))}")
print(f"Val images  : {len(os.listdir(VAL_DIR))}")
print(f"Test images : {len(os.listdir(TEST_DIR))}")

Train images: 2718
Val images  : 354
Test images : 238


In [ ]:
import cv2
from pathlib import Path

def load_image_and_labels(img_path, label_path, img_size=320):
    """Load image and convert YOLO labels to classification label"""
    img = cv2.imread(str(img_path))
    img = cv2.resize(img, (img_size, img_size))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0  # Normalize to 0-1

    # Get primary class from label file
    primary_class = 20  # default = 'object'
    if label_path.exists():
        with open(label_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        if lines:
            primary_class = int(lines[0].split()[0])

    return img.astype(np.float32), primary_class


def build_dataset(img_dir, label_dir, img_size=320):
    img_dir = Path(img_dir)
    label_dir = Path(label_dir)

    images = []
    labels = []

    img_files = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))

    for img_path in img_files:
        label_path = label_dir / (img_path.stem + '.txt')
        img, label = load_image_and_labels(img_path, label_path, img_size)
        images.append(img)
        labels.append(label)

    return np.array(images), np.array(labels)


print("Loading datasets...")
train_images, train_labels = build_dataset(
    f'{DATASET_PATH}/train/images',
    f'{DATASET_PATH}/train/labels'
)
val_images, val_labels = build_dataset(
    f'{DATASET_PATH}/val/images',
    f'{DATASET_PATH}/val/labels'
)
test_images, test_labels = build_dataset(
    f'{DATASET_PATH}/test/images',
    f'{DATASET_PATH}/test/labels'
)

print(f"Train: {train_images.shape}, Val: {val_images.shape}, Test: {test_images.shape}")

Loading datasets...
Train: (2718, 320, 320, 3), Val: (354, 320, 320, 3), Test: (238, 320, 320, 3)


In [ ]:
# ── Original code used a simple MNIST CNN ──────────────────────────────────
# CHANGED: Architecture redesigned for:
# - 320x320 RGB input (was 28x28 grayscale)
# - 21 navigation classes (was 10 digit classes)
# - MobileNetV2 backbone (lightweight, suitable for TinyML)
# - Still uses same pruning + quantization approach from original

def build_tinyml_model(input_shape=(320, 320, 3), num_classes=21):
    """
    MobileNetV2-based model for navigation object detection/classification
    Designed to be small enough for TinyML deployment after pruning + quantization
    """
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet',  # Pretrained weights — saves training time
        alpha=0.35           # Width multiplier — 0.35 = smallest version
    )

    # Freeze base model initially — fine-tune later
    base_model.trainable = False

    model = keras.Sequential([
        base_model,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(num_classes)  # Output: 21 navigation classes
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    model.summary()
    return model


model = build_tinyml_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NC)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_0.35_224 (Func  (None, 10, 10, 1280)      410208    
 tional)                                                         
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 128)               163968    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 21)                2709      
                                                                 
Total params: 576885 (2.20 MB)
Trainable params: 166677 

In [ ]:
# ── Same approach as original: train baseline first, then prune ────────────

callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
    keras.callbacks.ModelCheckpoint(
        f'{MODEL_SAVE_PATH}/baseline_best.keras',
        save_best_only=True
    )
]

print("Training baseline model...")
history = model.fit(
    train_images, train_labels,
    epochs=EPOCHS,
    batch_size=BATCH,
    validation_data=(val_images, val_labels),
    callbacks=callbacks
)

_, baseline_accuracy = model.evaluate(test_images, test_labels, verbose=0)
print(f"\nBaseline test accuracy: {baseline_accuracy:.4f}")

# Save baseline — same as original
_, keras_file = tempfile.mkstemp('.keras')
model.save(keras_file)
print(f"Saved baseline model to: {keras_file}")

Training baseline model...
Epoch 1/30
170/170 [==============================] - 108s 615ms/step - loss: 1.5589 - accuracy: 0.5276 - val_loss: 0.9786 - val_accuracy: 0.7232 - lr: 0.0010
Epoch 2/30
170/170 [==============================] - 97s 572ms/step - loss: 1.0158 - accuracy: 0.6711 - val_loss: 0.8430 - val_accuracy: 0.7119 - lr: 0.0010
Epoch 3/30
170/170 [==============================] - 100s 591ms/step - loss: 0.8256 - accuracy: 0.7248 - val_loss: 0.7704 - val_accuracy: 0.7458 - lr: 0.0010
Epoch 4/30
170/170 [==============================] - 98s 573ms/step - loss: 0.7188 - accuracy: 0.7675 - val_loss: 0.7655 - val_accuracy: 0.7401 - lr: 0.0010
Epoch 5/30
170/170 [==============================] - 107s 631ms/step - loss: 0.6317 - accuracy: 0.7862 - val_loss: 0.7482 - val_accuracy: 0.7514 - lr: 0.0010
Epoch 6/30
170/170 [==============================] - 102s 601ms/step - loss: 0.5659 - accuracy: 0.8164 - val_loss: 0.7292 - val_accuracy: 0.7429 - lr: 0.0010
Epoch 7/30
170/170 [=

In [ ]:
# ── SAME as original: magnitude-based pruning ─────────────────────────────
# Original pruned from 50% → 80% sparsity over 2 epochs
# CHANGED: More epochs since dataset is larger and task is harder

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

batch_size      = BATCH
pruning_epochs  = 5       # Original used 2 — increased for better convergence
validation_split = 0.1

num_images = train_images.shape[0] * (1 - validation_split)
end_step   = np.ceil(num_images / batch_size).astype(np.int32) * pruning_epochs

# Same pruning schedule as original
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.50,
        final_sparsity=0.80,
        begin_step=0,
        end_step=end_step
    )
}

model_for_pruning = prune_low_magnitude(model, **pruning_params)

model_for_pruning.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),  # Lower LR for fine-tuning
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model_for_pruning.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_0.35_224 (Func  (None, 10, 10, 1280)      768972    
 tional)                                                         
                                                                 
 prune_low_magnitude_global  (None, 1280)              1         
 _average_pooling2d (PruneL                                      
 owMagnitude)                                                    
                                                                 
 prune_low_magnitude_dense   (None, 128)               327810    
 (PruneLowMagnitude)                                             
                                                                 
 prune_low_magnitude_dropou  (None, 128)               1         
 t (PruneLowMagnitude)                                           
                                                        

In [ ]:
logdir = tempfile.mkdtemp()

# Same callbacks as original
callbacks_pruning = [
    tfmot.sparsity.keras.UpdatePruningStep(),
    tfmot.sparsity.keras.PruningSummaries(log_dir=logdir),
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
]

model_for_pruning.fit(
    train_images, train_labels,
    batch_size=batch_size,
    epochs=pruning_epochs,
    validation_data=(val_images, val_labels),
    callbacks=callbacks_pruning
)

_, pruned_accuracy = model_for_pruning.evaluate(test_images, test_labels, verbose=0)
print(f"\nBaseline accuracy : {baseline_accuracy:.4f}")
print(f"Pruned accuracy   : {pruned_accuracy:.4f}")
print(f"Accuracy drop     : {baseline_accuracy - pruned_accuracy:.4f}")

NameError: name 'tempfile' is not defined